In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init
import numpy as np
import os
import ntpath
import argparse
from threading import Thread
from heapq import heappop, heapify
import functools
from torch.optim import lr_scheduler
import glob
from torch.utils.data import Dataset, DataLoader

# --- Mocks for missing util.util functions ---
def print_network(net):
    num_params = 0
    for param in net.parameters():
        num_params += param.numel()
    print('Total number of parameters: %d' % num_params)

def seg_accuracy(pred, label, mesh):
    # Placeholder for segmentation accuracy (not used in binary classification)
    return 0

# --- Helper for padding ---
def fixed_division(to_div, epsilon):
    if epsilon == 0:
        to_div[to_div == 0] = 0.1
    else:
        to_div += epsilon
    return to_div

In [42]:
class MeshUnion:
    def __init__(self, n, device=torch.device('cpu')):
        self.__size = n
        self.rebuild_features = self.rebuild_features_average
        self.groups = torch.eye(n, device=device)

    def union(self, source, target):
        self.groups[target, :] += self.groups[source, :]

    def remove_group(self, index):
        return

    def get_group(self, edge_key):
        return self.groups[edge_key, :]

    def get_occurrences(self):
        return torch.sum(self.groups, 0)

    def get_groups(self, tensor_mask):
        self.groups = torch.clamp(self.groups, 0, 1)
        return self.groups[tensor_mask, :]

    def rebuild_features_average(self, features, mask, target_edges):
        self.prepare_groups(features, mask)
        fe = torch.matmul(features.squeeze(-1), self.groups)
        occurrences = torch.sum(self.groups, 0).expand(fe.shape)
        fe = fe / occurrences
        padding_b = target_edges - fe.shape[1]
        if padding_b > 0:
            padding_b = nn.ConstantPad2d((0, padding_b, 0, 0), 0)
            fe = padding_b(fe)
        return fe

    def prepare_groups(self, features, mask):
        tensor_mask = torch.from_numpy(mask)
        self.groups = torch.clamp(self.groups[tensor_mask, :], 0, 1).transpose_(1, 0)
        padding_a = features.shape[1] - self.groups.shape[0]
        if padding_a > 0:
            padding_a = nn.ConstantPad2d((0, 0, 0, padding_a), 0)
            self.groups = padding_a(self.groups)

# --- Mesh Preparation Functions ---
def fill_mesh(mesh2fill, file: str, opt):
    # Simplified to avoid caching complex paths for the notebook example
    # In a full run, you might want to restore the caching logic
    mesh_data = from_scratch(file, opt)
    mesh2fill.vs = mesh_data.vs
    mesh2fill.edges = mesh_data.edges
    mesh2fill.gemm_edges = mesh_data.gemm_edges
    mesh2fill.edges_count = int(mesh_data.edges_count)
    mesh2fill.ve = mesh_data.ve
    mesh2fill.v_mask = mesh_data.v_mask
    mesh2fill.filename = str(mesh_data.filename)
    mesh2fill.edge_lengths = mesh_data.edge_lengths
    mesh2fill.edge_areas = mesh_data.edge_areas
    mesh2fill.features = mesh_data.features
    mesh2fill.sides = mesh_data.sides

def from_scratch(file, opt):
    class MeshPrep:
        def __getitem__(self, item):
            return eval('self.' + item)
    mesh_data = MeshPrep()
    mesh_data.vs = mesh_data.edges = None
    mesh_data.gemm_edges = mesh_data.sides = None
    mesh_data.edges_count = None
    mesh_data.ve = None
    mesh_data.v_mask = None
    mesh_data.filename = 'unknown'
    mesh_data.edge_lengths = None
    mesh_data.edge_areas = []
    mesh_data.vs, faces = fill_from_file(mesh_data, file)
    mesh_data.v_mask = np.ones(len(mesh_data.vs), dtype=bool)
    faces, face_areas = remove_non_manifolds(mesh_data, faces)
    build_gemm(mesh_data, faces, face_areas)
    mesh_data.features = extract_features(mesh_data)
    return mesh_data

def fill_from_file(mesh, file):
    mesh.filename = ntpath.split(file)[1]
    mesh.fullfilename = file
    vs, faces = [], []
    f = open(file)
    for line in f:
        line = line.strip()
        splitted_line = line.split()
        if not splitted_line:
            continue
        elif splitted_line[0] == 'v':
            vs.append([float(v) for v in splitted_line[1:4]])
        elif splitted_line[0] == 'f':
            face_vertex_ids = [int(c.split('/')[0]) for c in splitted_line[1:]]
            assert len(face_vertex_ids) == 3
            face_vertex_ids = [(ind - 1) if (ind >= 0) else (len(vs) + ind) for ind in face_vertex_ids]
            faces.append(face_vertex_ids)
    f.close()
    vs = np.asarray(vs)
    faces = np.asarray(faces, dtype=int)
    assert np.logical_and(faces >= 0, faces < len(vs)).all()
    return vs, faces

def remove_non_manifolds(mesh, faces):
    mesh.ve = [[] for _ in mesh.vs]
    edges_set = set()
    mask = np.ones(len(faces), dtype=bool)
    _, face_areas = compute_face_normals_and_areas(mesh, faces)
    for face_id, face in enumerate(faces):
        if face_areas[face_id] == 0:
            mask[face_id] = False
            continue
        faces_edges = []
        is_manifold = False
        for i in range(3):
            cur_edge = (face[i], face[(i + 1) % 3])
            if cur_edge in edges_set:
                is_manifold = True
                break
            else:
                faces_edges.append(cur_edge)
        if is_manifold:
            mask[face_id] = False
        else:
            for idx, edge in enumerate(faces_edges):
                edges_set.add(edge)
    return faces[mask], face_areas[mask]

def build_gemm(mesh, faces, face_areas):
    mesh.ve = [[] for _ in mesh.vs]
    edge_nb = []
    sides = []
    edge2key = dict()
    edges = []
    edges_count = 0
    nb_count = []
    for face_id, face in enumerate(faces):
        faces_edges = []
        for i in range(3):
            cur_edge = (face[i], face[(i + 1) % 3])
            faces_edges.append(cur_edge)
        for idx, edge in enumerate(faces_edges):
            edge = tuple(sorted(list(edge)))
            faces_edges[idx] = edge
            if edge not in edge2key:
                edge2key[edge] = edges_count
                edges.append(list(edge))
                edge_nb.append([-1, -1, -1, -1])
                sides.append([-1, -1, -1, -1])
                mesh.ve[edge[0]].append(edges_count)
                mesh.ve[edge[1]].append(edges_count)
                mesh.edge_areas.append(0)
                nb_count.append(0)
                edges_count += 1
            mesh.edge_areas[edge2key[edge]] += face_areas[face_id] / 3
        for idx, edge in enumerate(faces_edges):
            edge_key = edge2key[edge]
            edge_nb[edge_key][nb_count[edge_key]] = edge2key[faces_edges[(idx + 1) % 3]]
            edge_nb[edge_key][nb_count[edge_key] + 1] = edge2key[faces_edges[(idx + 2) % 3]]
            nb_count[edge_key] += 2
        for idx, edge in enumerate(faces_edges):
            edge_key = edge2key[edge]
            sides[edge_key][nb_count[edge_key] - 2] = nb_count[edge2key[faces_edges[(idx + 1) % 3]]] - 1
            sides[edge_key][nb_count[edge_key] - 1] = nb_count[edge2key[faces_edges[(idx + 2) % 3]]] - 2
    mesh.edges = np.array(edges, dtype=np.int32)
    mesh.gemm_edges = np.array(edge_nb, dtype=np.int64)
    mesh.sides = np.array(sides, dtype=np.int64)
    mesh.edges_count = edges_count
    mesh.edge_areas = np.array(mesh.edge_areas, dtype=np.float32) / np.sum(face_areas)

def compute_face_normals_and_areas(mesh, faces):
    face_normals = np.cross(mesh.vs[faces[:, 1]] - mesh.vs[faces[:, 0]], mesh.vs[faces[:, 2]] - mesh.vs[faces[:, 1]])
    face_areas = np.sqrt((face_normals ** 2).sum(axis=1))
    face_normals /= face_areas[:, np.newaxis]
    face_areas *= 0.5
    # The return statement was likely malformed in the previous copy/paste
    return face_normals, face_areas

In [43]:
class Mesh:
    def __init__(self, file=None, opt=None, hold_history=False, export_folder=''):
        self.vs = self.v_mask = self.filename = self.features = self.edge_areas = None
        self.edges = self.gemm_edges = self.sides = None
        self.pool_count = 0
        fill_mesh(self, file, opt)
        self.export_folder = export_folder
        self.history_data = None
        if hold_history:
            self.init_history()

    def extract_features(self):
        return self.features

    def merge_vertices(self, edge_id):
        self.remove_edge(edge_id)
        edge = self.edges[edge_id]
        v_a = self.vs[edge[0]]
        v_b = self.vs[edge[1]]
        v_a.__iadd__(v_b)
        v_a.__itruediv__(2)
        self.v_mask[edge[1]] = False
        mask = self.edges == edge[1]
        self.ve[edge[0]].extend(self.ve[edge[1]])
        self.edges[mask] = edge[0]

    def remove_vertex(self, v):
        self.v_mask[v] = False

    def remove_edge(self, edge_id):
        vs = self.edges[edge_id]
        for v in vs:
            if edge_id in self.ve[v]:
                self.ve[v].remove(edge_id)

    def clean(self, edges_mask, groups):
        edges_mask = edges_mask.astype(bool)
        torch_mask = torch.from_numpy(edges_mask.copy())
        self.gemm_edges = self.gemm_edges[edges_mask]
        self.edges = self.edges[edges_mask]
        self.sides = self.sides[edges_mask]
        new_ve = []
        edges_mask = np.concatenate([edges_mask, [False]])
        new_indices = np.zeros(edges_mask.shape[0], dtype=np.int32)
        new_indices[-1] = -1
        new_indices[edges_mask] = np.arange(0, np.ma.where(edges_mask)[0].shape[0])
        self.gemm_edges[:, :] = new_indices[self.gemm_edges[:, :]]
        for v_index, ve in enumerate(self.ve):
            update_ve = []
            for e in ve:
                update_ve.append(new_indices[e])
            new_ve.append(update_ve)
        self.ve = new_ve
        self.__clean_history(groups, torch_mask)
        self.pool_count += 1

    def init_history(self):
        self.history_data = {
                               'groups': [],
                               'gemm_edges': [self.gemm_edges.copy()],
                               'occurrences': [],
                               'old2current': np.arange(self.edges_count, dtype=np.int32),
                               'current2old': np.arange(self.edges_count, dtype=np.int32),
                               'edges_mask': [torch.ones(self.edges_count,dtype=torch.bool)],
                               'edges_count': [self.edges_count],
                              }
        if self.export_folder:
            self.history_data['collapses'] = MeshUnion(self.edges_count)

    def union_groups(self, source, target):
        if self.export_folder and self.history_data:
            self.history_data['collapses'].union(self.history_data['current2old'][source], self.history_data['current2old'][target])
        return

    def remove_group(self, index):
        if self.history_data is not None:
            self.history_data['edges_mask'][-1][self.history_data['current2old'][index]] = 0
            self.history_data['old2current'][self.history_data['current2old'][index]] = -1
            if self.export_folder:
                self.history_data['collapses'].remove_group(self.history_data['current2old'][index])

    def get_groups(self):
        return self.history_data['groups'].pop()

    def get_occurrences(self):
        return self.history_data['occurrences'].pop()
    
    def __clean_history(self, groups, pool_mask):
        if self.history_data is not None:
            mask = self.history_data['old2current'] != -1
            self.history_data['old2current'][mask] = np.arange(self.edges_count, dtype=np.int32)
            self.history_data['current2old'][0: self.edges_count] = np.ma.where(mask)[0]
            if self.export_folder != '':
                self.history_data['edges_mask'].append(self.history_data['edges_mask'][-1].clone())
            self.history_data['occurrences'].append(groups.get_occurrences())
            self.history_data['groups'].append(groups.get_groups(pool_mask))
            self.history_data['gemm_edges'].append(self.gemm_edges.copy())
            self.history_data['edges_count'].append(self.edges_count)
    
    def unroll_gemm(self):
        self.history_data['gemm_edges'].pop()
        self.gemm_edges = self.history_data['gemm_edges'][-1]
        self.history_data['edges_count'].pop()
        self.edges_count = self.history_data['edges_count'][-1]

    def get_edge_areas(self):
        return self.edge_areas

In [50]:
import numpy as np

def extract_features(mesh):
    # 1. Get the 3D coordinates for the start and end of every edge
    # mesh.edges[:, 0] = indices of the first vertex
    # mesh.edges[:, 1] = indices of the second vertex
    p1 = mesh.vs[mesh.edges[:, 0]]
    p2 = mesh.vs[mesh.edges[:, 1]]
    
    # 2. Calculate Edge Lengths (Euclidean distance)
    # We use numpy's norm function across axis 1 (the x,y,z dimension)
    edge_lengths = np.linalg.norm(p1 - p2, axis=1)
    
    # 3. Initialize features array
    # Shape: (Number of Edges, 5) to match standard input_nc=5
    features = np.zeros((mesh.edges.shape[0], 5))
    
    # 4. Set the first feature to the edge length
    features[:, 0] = edge_lengths
    
    # (Optional) If you want to normalize the lengths relative to the average:
    # features[:, 0] = edge_lengths / np.mean(edge_lengths)
    
    return features

In [51]:
# REPLACE CELL 4 WITH THIS
import torch
import torch.nn as nn
import torch.nn.functional as F
from threading import Thread
from heapq import heappop, heapify

# --- MeshConv ---
class MeshConv(nn.Module):
    def __init__(self, in_channels, out_channels, k=5, bias=True):
        super(MeshConv, self).__init__()
        self.conv = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=(1, k), bias=bias)
        self.k = k

    def __call__(self, edge_f, mesh):
        return self.forward(edge_f, mesh)

    def forward(self, x, mesh):
        x = x.squeeze(-1)
        G = torch.cat([self.pad_gemm(i, x.shape[2], x.device) for i in mesh], 0)
        G = self.create_GeMM(x, G)
        x = self.conv(G)
        return x

    def flatten_gemm_inds(self, Gi):
        (b, ne, nn) = Gi.shape
        ne += 1
        batch_n = torch.floor(torch.arange(b * ne, device=Gi.device).float() / ne).view(b, ne)
        add_fac = batch_n * ne
        add_fac = add_fac.view(b, ne, 1)
        add_fac = add_fac.repeat(1, 1, nn)
        Gi = Gi.float() + add_fac[:, 1:, :]
        return Gi

    def create_GeMM(self, x, Gi):
        Gishape = Gi.shape
        padding = torch.zeros((x.shape[0], x.shape[1], 1), requires_grad=True, device=x.device)
        x = torch.cat((padding, x), dim=2)
        Gi = Gi + 1 
        Gi_flat = self.flatten_gemm_inds(Gi)
        Gi_flat = Gi_flat.view(-1).long()
        odim = x.shape
        x = x.permute(0, 2, 1).contiguous()
        x = x.view(odim[0] * odim[2], odim[1])
        f = torch.index_select(x, dim=0, index=Gi_flat)
        f = f.view(Gishape[0], Gishape[1], Gishape[2], -1)
        f = f.permute(0, 3, 1, 2)
        x_1 = f[:, :, :, 1] + f[:, :, :, 3]
        x_2 = f[:, :, :, 2] + f[:, :, :, 4]
        x_3 = torch.abs(f[:, :, :, 1] - f[:, :, :, 3])
        x_4 = torch.abs(f[:, :, :, 2] - f[:, :, :, 4])
        f = torch.stack([f[:, :, :, 0], x_1, x_2, x_3, x_4], dim=3)
        return f

    def pad_gemm(self, m, xsz, device):
        padded_gemm = torch.tensor(m.gemm_edges, device=device).float()
        padded_gemm = padded_gemm.requires_grad_()
        padded_gemm = torch.cat((torch.arange(m.edges_count, device=device).float().unsqueeze(1), padded_gemm), dim=1)
        padded_gemm = F.pad(padded_gemm, (0, 0, 0, xsz - m.edges_count), "constant", 0)
        padded_gemm = padded_gemm.unsqueeze(0)
        return padded_gemm

# --- MeshPool (Fixed Names) ---
class MeshPool(nn.Module):
    def __init__(self, target, multi_thread=False):
        super(MeshPool, self).__init__()
        self.out_target = target
        self.multi_thread = multi_thread
        self.fe = None
        self.updated_fe = None
        self.meshes = None
        self.merge_edges = [-1, -1]

    def __call__(self, fe, meshes):
        return self.forward(fe, meshes)

    def forward(self, fe, meshes):
        self.updated_fe = [[] for _ in range(len(meshes))]
        self.fe = fe
        self.meshes = meshes
        # Using simple loop to avoid threading issues in notebooks
        for mesh_index in range(len(meshes)):
            self._pool_main(mesh_index)
            
        out_features = torch.cat(self.updated_fe).view(len(meshes), -1, self.out_target)
        return out_features

    def _pool_main(self, mesh_index):
        mesh = self.meshes[mesh_index]
        # Changed from __build_queue to _build_queue
        queue = self._build_queue(self.fe[mesh_index, :, :mesh.edges_count], mesh.edges_count)
        mask = np.ones(mesh.edges_count, dtype=np.bool_)
        edge_groups = MeshUnion(mesh.edges_count, self.fe.device)
        while mesh.edges_count > self.out_target:
            value, edge_id = heappop(queue)
            edge_id = int(edge_id)
            if mask[edge_id]:
                self._pool_edge(mesh, edge_id, mask, edge_groups)
        mesh.clean(mask, edge_groups)
        fe = edge_groups.rebuild_features(self.fe[mesh_index], mask, self.out_target)
        self.updated_fe[mesh_index] = fe

    def _pool_edge(self, mesh, edge_id, mask, edge_groups):
        if self.has_boundaries(mesh, edge_id):
            return False
        elif self._clean_side(mesh, edge_id, mask, edge_groups, 0)\
            and self._clean_side(mesh, edge_id, mask, edge_groups, 2) \
            and self._is_one_ring_valid(mesh, edge_id):
            self.merge_edges[0] = self._pool_side(mesh, edge_id, mask, edge_groups, 0)
            self.merge_edges[1] = self._pool_side(mesh, edge_id, mask, edge_groups, 2)
            mesh.merge_vertices(edge_id)
            mask[edge_id] = False
            self._remove_group(mesh, edge_groups, edge_id)
            mesh.edges_count -= 1
            return True
        else:
            return False

    def _clean_side(self, mesh, edge_id, mask, edge_groups, side):
        if mesh.edges_count <= self.out_target:
            return False
        invalid_edges = self._get_invalids(mesh, edge_id, edge_groups, side)
        while len(invalid_edges) != 0 and mesh.edges_count > self.out_target:
            self._remove_triplete(mesh, mask, edge_groups, invalid_edges)
            if mesh.edges_count <= self.out_target:
                return False
            if self.has_boundaries(mesh, edge_id):
                return False
            invalid_edges = self._get_invalids(mesh, edge_id, edge_groups, side)
        return True

    @staticmethod
    def has_boundaries(mesh, edge_id):
        for edge in mesh.gemm_edges[edge_id]:
            if edge == -1 or -1 in mesh.gemm_edges[edge]:
                return True
        return False

    @staticmethod
    def _is_one_ring_valid(mesh, edge_id):
        v_a = set(mesh.edges[mesh.ve[mesh.edges[edge_id, 0]]].reshape(-1))
        v_b = set(mesh.edges[mesh.ve[mesh.edges[edge_id, 1]]].reshape(-1))
        shared = v_a & v_b - set(mesh.edges[edge_id])
        return len(shared) == 2

    def _pool_side(self, mesh, edge_id, mask, edge_groups, side):
        info = self._get_face_info(mesh, edge_id, side)
        key_a, key_b, side_a, side_b, _, other_side_b, _, other_keys_b = info
        self._redirect_edges(mesh, key_a, side_a - side_a % 2, other_keys_b[0], mesh.sides[key_b, other_side_b])
        self._redirect_edges(mesh, key_a, side_a - side_a % 2 + 1, other_keys_b[1], mesh.sides[key_b, other_side_b + 1])
        self._union_groups(mesh, edge_groups, key_b, key_a)
        self._union_groups(mesh, edge_groups, edge_id, key_a)
        mask[key_b] = False
        self._remove_group(mesh, edge_groups, key_b)
        mesh.remove_edge(key_b)
        mesh.edges_count -= 1
        return key_a

    @staticmethod
    def _get_invalids(mesh, edge_id, edge_groups, side):
        info = MeshPool._get_face_info(mesh, edge_id, side)
        key_a, key_b, side_a, side_b, other_side_a, other_side_b, other_keys_a, other_keys_b = info
        shared_items = MeshPool._get_shared_items(other_keys_a, other_keys_b)
        if len(shared_items) == 0:
            return []
        else:
            assert (len(shared_items) == 2)
            middle_edge = other_keys_a[shared_items[0]]
            update_key_a = other_keys_a[1 - shared_items[0]]
            update_key_b = other_keys_b[1 - shared_items[1]]
            update_side_a = mesh.sides[key_a, other_side_a + 1 - shared_items[0]]
            update_side_b = mesh.sides[key_b, other_side_b + 1 - shared_items[1]]
            MeshPool._redirect_edges(mesh, edge_id, side, update_key_a, update_side_a)
            MeshPool._redirect_edges(mesh, edge_id, side + 1, update_key_b, update_side_b)
            MeshPool._redirect_edges(mesh, update_key_a, MeshPool._get_other_side(update_side_a), update_key_b, MeshPool._get_other_side(update_side_b))
            MeshPool._union_groups(mesh, edge_groups, key_a, edge_id)
            MeshPool._union_groups(mesh, edge_groups, key_b, edge_id)
            MeshPool._union_groups(mesh, edge_groups, key_a, update_key_a)
            MeshPool._union_groups(mesh, edge_groups, middle_edge, update_key_a)
            MeshPool._union_groups(mesh, edge_groups, key_b, update_key_b)
            MeshPool._union_groups(mesh, edge_groups, middle_edge, update_key_b)
            return [key_a, key_b, middle_edge]

    @staticmethod
    def _redirect_edges(mesh, edge_a_key, side_a, edge_b_key, side_b):
        mesh.gemm_edges[edge_a_key, side_a] = edge_b_key
        mesh.gemm_edges[edge_b_key, side_b] = edge_a_key
        mesh.sides[edge_a_key, side_a] = side_b
        mesh.sides[edge_b_key, side_b] = side_a

    @staticmethod
    def _get_shared_items(list_a, list_b):
        shared_items = []
        for i in range(len(list_a)):
            for j in range(len(list_b)):
                if list_a[i] == list_b[j]:
                    shared_items.extend([i, j])
        return shared_items

    @staticmethod
    def _get_other_side(side):
        return side + 1 - 2 * (side % 2)

    @staticmethod
    def _get_face_info(mesh, edge_id, side):
        key_a = mesh.gemm_edges[edge_id, side]
        key_b = mesh.gemm_edges[edge_id, side + 1]
        side_a = mesh.sides[edge_id, side]
        side_b = mesh.sides[edge_id, side + 1]
        other_side_a = (side_a - (side_a % 2) + 2) % 4
        other_side_b = (side_b - (side_b % 2) + 2) % 4
        other_keys_a = [mesh.gemm_edges[key_a, other_side_a], mesh.gemm_edges[key_a, other_side_a + 1]]
        other_keys_b = [mesh.gemm_edges[key_b, other_side_b], mesh.gemm_edges[key_b, other_side_b + 1]]
        return key_a, key_b, side_a, side_b, other_side_a, other_side_b, other_keys_a, other_keys_b

    @staticmethod
    def _remove_triplete(mesh, mask, edge_groups, invalid_edges):
        vertex = set(mesh.edges[invalid_edges[0]])
        for edge_key in invalid_edges:
            vertex &= set(mesh.edges[edge_key])
            mask[edge_key] = False
            MeshPool._remove_group(mesh, edge_groups, edge_key)
        mesh.edges_count -= 3
        vertex = list(vertex)
        mesh.remove_vertex(vertex[0])

    # Renamed from __build_queue to _build_queue to match call in _pool_main
    def _build_queue(self, features, edges_count):
        squared_magnitude = torch.sum(features * features, 0)
        if squared_magnitude.shape[-1] != 1:
            squared_magnitude = squared_magnitude.unsqueeze(-1)
        edge_ids = torch.arange(edges_count, device=squared_magnitude.device, dtype=torch.float32).unsqueeze(-1)
        heap = torch.cat((squared_magnitude, edge_ids), dim=-1).tolist()
        heapify(heap)
        return heap

    @staticmethod
    def _union_groups(mesh, edge_groups, source, target):
        edge_groups.union(source, target)
        mesh.union_groups(source, target)

    @staticmethod
    def _remove_group(mesh, edge_groups, index):
        edge_groups.remove_group(index)
        mesh.remove_group(index)

# --- MeshUnpool ---
class MeshUnpool(nn.Module):
    def __init__(self, unroll_target):
        super(MeshUnpool, self).__init__()
        self.unroll_target = unroll_target

    def __call__(self, features, meshes):
        return self.forward(features, meshes)

    def pad_groups(self, group, unroll_start):
        start, end = group.shape
        padding_rows =  unroll_start - start
        padding_cols = self.unroll_target - end
        if padding_rows != 0 or padding_cols !=0:
            padding = nn.ConstantPad2d((0, padding_cols, 0, padding_rows), 0)
            group = padding(group)
        return group

    def pad_occurrences(self, occurrences):
        padding = self.unroll_target - occurrences.shape[0]
        if padding != 0:
            padding = nn.ConstantPad1d((0, padding), 1)
            occurrences = padding(occurrences)
        return occurrences

    def forward(self, features, meshes):
        batch_size, nf, edges = features.shape
        groups = [self.pad_groups(mesh.get_groups(), edges) for mesh in meshes]
        unroll_mat = torch.cat(groups, dim=0).view(batch_size, edges, -1)
        occurrences = [self.pad_occurrences(mesh.get_occurrences()) for mesh in meshes]
        occurrences = torch.cat(occurrences, dim=0).view(batch_size, 1, -1)
        occurrences = occurrences.expand(unroll_mat.shape)
        unroll_mat = unroll_mat / occurrences
        unroll_mat = unroll_mat.to(features.device)
        for mesh in meshes:
            mesh.unroll_gemm()
        return torch.matmul(features, unroll_mat)

In [52]:
def get_norm_layer(norm_type='instance', num_groups=1):
    if norm_type == 'batch':
        norm_layer = functools.partial(nn.BatchNorm2d, affine=True)
    elif norm_type == 'instance':
        norm_layer = functools.partial(nn.InstanceNorm2d, affine=False)
    elif norm_type == 'group':
        norm_layer = functools.partial(nn.GroupNorm, affine=True, num_groups=num_groups)
    elif norm_type == 'none':
        norm_layer = NoNorm
    return norm_layer

def get_norm_args(norm_layer, nfeats_list):
    if hasattr(norm_layer, '__name__') and norm_layer.__name__ == 'NoNorm':
        norm_args = [{'fake': True} for f in nfeats_list]
    elif norm_layer.func.__name__ == 'GroupNorm':
        norm_args = [{'num_channels': f} for f in nfeats_list]
    # FIX: Use 'in' to check for BatchNorm2d, BatchNorm1d, etc.
    elif 'BatchNorm' in norm_layer.func.__name__:
        norm_args = [{'num_features': f} for f in nfeats_list]
    # FIX: Add support for InstanceNorm
    elif 'InstanceNorm' in norm_layer.func.__name__:
        norm_args = [{'num_features': f} for f in nfeats_list]
    else:
        # Fallback for unexpected layers to prevent UnboundLocalError
        print(f"Warning: Unknown norm layer {norm_layer.func.__name__}, defaulting to num_features")
        norm_args = [{'num_features': f} for f in nfeats_list]
    return norm_args

class NoNorm(nn.Module):
    def __init__(self, fake=True):
        self.fake = fake
        super(NoNorm, self).__init__()
    def forward(self, x):
        return x

def init_weights(net, init_type, init_gain):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and (classname.find('Conv') != -1 or classname.find('Linear') != -1):
            if init_type == 'normal':
                init.normal_(m.weight.data, 0.0, init_gain)
            elif init_type == 'xavier':
                init.xavier_normal_(m.weight.data, gain=init_gain)
            elif init_type == 'kaiming':
                init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
        elif classname.find('BatchNorm2d') != -1:
            init.normal_(m.weight.data, 1.0, init_gain)
            init.constant_(m.bias.data, 0.0)
    net.apply(init_func)

def init_net(net, init_type, init_gain, gpu_ids):
    if len(gpu_ids) > 0:
        assert(torch.cuda.is_available())
        net.cuda(gpu_ids[0])
        net = torch.nn.DataParallel(net, gpu_ids)
    if init_type != 'none':
        init_weights(net, init_type, init_gain)
    return net

def define_classifier(input_nc, ncf, ninput_edges, nclasses, opt, gpu_ids, arch, init_type, init_gain):
    norm_layer = get_norm_layer(norm_type=opt.norm, num_groups=opt.num_groups)
    if arch == 'mconvnet':
        net = MeshConvNet(norm_layer, input_nc, ncf, nclasses, ninput_edges, opt.pool_res, opt.fc_n, opt.resblocks)
    else:
        raise NotImplementedError('Encoder model name [%s] is not recognized' % arch)
    return init_net(net, init_type, init_gain, gpu_ids)

def define_loss(opt):
    if opt.dataset_mode == 'classification':
        loss = torch.nn.CrossEntropyLoss()
    elif opt.dataset_mode == 'segmentation':
        loss = torch.nn.CrossEntropyLoss(ignore_index=-1)
    return loss

class MeshConvNet(nn.Module):
    def __init__(self, norm_layer, nf0, conv_res, nclasses, input_res, pool_res, fc_n, nresblocks=3):
        super(MeshConvNet, self).__init__()
        self.k = [nf0] + conv_res
        self.res = [input_res] + pool_res
        norm_args = get_norm_args(norm_layer, self.k[1:])
        for i, ki in enumerate(self.k[:-1]):
            setattr(self, 'conv{}'.format(i), MResConv(ki, self.k[i + 1], nresblocks))
            setattr(self, 'norm{}'.format(i), norm_layer(**norm_args[i]))
            setattr(self, 'pool{}'.format(i), MeshPool(self.res[i + 1]))
        self.gp = torch.nn.AvgPool1d(self.res[-1])
        self.fc1 = nn.Linear(self.k[-1], fc_n)
        self.fc2 = nn.Linear(fc_n, nclasses)

    def forward(self, x, mesh):
        for i in range(len(self.k) - 1):
            x = getattr(self, 'conv{}'.format(i))(x, mesh)
            x = F.relu(getattr(self, 'norm{}'.format(i))(x))
            x = getattr(self, 'pool{}'.format(i))(x, mesh)
        x = self.gp(x)
        x = x.view(-1, self.k[-1])
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class MResConv(nn.Module):
    def __init__(self, in_channels, out_channels, skips=1):
        super(MResConv, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.skips = skips
        self.conv0 = MeshConv(self.in_channels, self.out_channels, bias=False)
        for i in range(self.skips):
            setattr(self, 'bn{}'.format(i + 1), nn.BatchNorm2d(self.out_channels))
            setattr(self, 'conv{}'.format(i + 1), MeshConv(self.out_channels, self.out_channels, bias=False))

    def forward(self, x, mesh):
        x = self.conv0(x, mesh)
        x1 = x
        for i in range(self.skips):
            x = getattr(self, 'bn{}'.format(i + 1))(F.relu(x))
            x = getattr(self, 'conv{}'.format(i + 1))(x, mesh)
        x += x1
        x = F.relu(x)
        return x

In [53]:
class ClassifierModel:
    def __init__(self, opt):
        self.opt = opt
        self.gpu_ids = opt.gpu_ids
        self.is_train = opt.is_train
        self.device = torch.device('cuda:{}'.format(self.gpu_ids[0])) if self.gpu_ids else torch.device('cpu')
        self.save_dir = opt.checkpoints_dir
        self.optimizer = None
        self.edge_features = None
        self.labels = None
        self.mesh = None
        self.nclasses = opt.nclasses
        self.net = define_classifier(opt.input_nc, opt.ncf, opt.ninput_edges, opt.nclasses, opt,
                                              self.gpu_ids, opt.arch, opt.init_type, opt.init_gain)
        self.net.train(self.is_train)
        self.criterion = define_loss(opt).to(self.device)
        if self.is_train:
            self.optimizer = torch.optim.Adam(self.net.parameters(), lr=opt.lr, betas=(opt.beta1, 0.999))
            # Simple scheduler to avoid import complexity
            self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=opt.lr_decay_iters, gamma=0.1)
            print_network(self.net)

    def set_input(self, data):
        input_edge_features = torch.from_numpy(data['edge_features']).float()
        labels = torch.from_numpy(data['label']).long()
        self.edge_features = input_edge_features.to(self.device).requires_grad_(self.is_train)
        self.labels = labels.to(self.device)
        self.mesh = data['mesh']

    def forward(self):
        out = self.net(self.edge_features, self.mesh)
        return out

    def backward(self, out):
        self.loss = self.criterion(out, self.labels)
        self.loss.backward()

    def optimize_parameters(self):
        self.optimizer.zero_grad()
        out = self.forward()
        self.backward(out)
        self.optimizer.step()

    def update_learning_rate(self):
        self.scheduler.step()
        lr = self.optimizer.param_groups[0]['lr']
        print('learning rate = %.7f' % lr)

    def test(self):
        with torch.no_grad():
            out = self.forward()
            pred_class = out.data.max(1)[1]
            correct = self.get_accuracy(pred_class, self.labels)
        return correct, len(self.labels)

    def get_accuracy(self, pred, labels):
        return pred.eq(labels).sum()

In [54]:
class TrainOptions:
    def parse(self):
        parser = argparse.ArgumentParser()
        # Paths
        parser.add_argument('--dataroot', type=str, default='./datasets/my_project', help='path to data')
        parser.add_argument('--name', type=str, default='mesh_classifier', help='experiment name')
        parser.add_argument('--checkpoints_dir', type=str, default='./checkpoints', help='save dir')
        parser.add_argument('--gpu_ids', type=str, default='0', help='gpu ids: e.g. 0  0,1,2, 0,2. use -1 for CPU')
        
        # Model Params
        parser.add_argument('--nclasses', type=int, default=2, help='num classes')
        parser.add_argument('--input_nc', type=int, default=5, help='input features')
        parser.add_argument('--ncf', type=int, nargs='+', default=[16, 32, 32], help='conv filters')
        parser.add_argument('--ninput_edges', type=int, default=750, help='target edges')
        parser.add_argument('--pool_res', type=int, nargs='+', default=[1140, 780, 580], help='pooling res')
        parser.add_argument('--fc_n', type=int, default=100, help='fully connected size')
        parser.add_argument('--resblocks', type=int, default=0, help='resblocks')
        parser.add_argument('--arch', type=str, default='mconvnet', help='architecture')
        parser.add_argument('--norm', type=str, default='batch', help='norm type')
        parser.add_argument('--num_groups', type=int, default=16, help='num groups')
        parser.add_argument('--init_type', type=str, default='normal', help='init type')
        parser.add_argument('--init_gain', type=float, default=0.02, help='init gain')
        
        # Train Params
        parser.add_argument('--dataset_mode', type=str, default='classification')
        parser.add_argument('--is_train', type=bool, default=True)
        parser.add_argument('--epoch_count', type=int, default=1)
        parser.add_argument('--niter', type=int, default=100)
        parser.add_argument('--niter_decay', type=int, default=100)
        parser.add_argument('--lr', type=float, default=0.0002)
        parser.add_argument('--beta1', type=float, default=0.5)
        parser.add_argument('--lr_decay_iters', type=int, default=50)
        parser.add_argument('--num_aug', type=int, default=1)
        
        opt, _ = parser.parse_known_args()
        
        opt.gpu_ids = [int(x) for x in opt.gpu_ids.split(',')]
        if '-1' in opt.gpu_ids:
            opt.gpu_ids = []
            
        return opt

class MeshDataset(Dataset):
    def __init__(self, opt):
        self.opt = opt
        self.root = opt.dataroot
        self.paths = []
        self.labels = []
        
        # Assume folder structure: dataroot/class_name/*.obj
        classes = sorted([d for d in os.listdir(self.root) if os.path.isdir(os.path.join(self.root, d))])
        print(f"Found classes: {classes}")
        
        for i, class_name in enumerate(classes):
            class_dir = os.path.join(self.root, class_name)
            files = sorted(glob.glob(os.path.join(class_dir, '*.obj')))
            for f in files:
                self.paths.append(f)
                self.labels.append(i) # 0 for first class, 1 for second
        
        print(f"Total samples: {len(self.paths)}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        label = self.labels[index]
        mesh = Mesh(file=path, opt=self.opt, hold_history=True, export_folder=self.opt.checkpoints_dir)
        meta = {
            'mesh': mesh,
            'label': label,
            'edge_features': mesh.extract_features()
        }
        return meta

def collate_fn(batch):
    edge_features = np.array([b['edge_features'] for b in batch])
    labels = np.array([b['label'] for b in batch])
    meshes = [b['mesh'] for b in batch]
    return {'edge_features': edge_features, 'label': labels, 'mesh': meshes}

In [55]:
if __name__ == '__main__':
    # 1. Parse Options
    opt = TrainOptions().parse()
    opt.niter = 10
    opt.niter_decay = 10
    # OVERRIDE DEFAULTS HERE IF NEEDED
    opt.dataroot = '/Users/ahashganeshamoorthy/Documents/Projects/Radiel/neuro/prototypes/V0/dataset/train' 
    
    # 2. Load Data
    dataset = MeshDataset(opt)
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
    opt.gpu_ids = []
    # 3. Create Model
    model = ClassifierModel(opt)
    print('Model created.')
    
    # 4. Training Loop
    total_steps = 0
    for epoch in range(opt.epoch_count, opt.niter + opt.niter_decay + 1):
        epoch_iter = 0
        for i, data in enumerate(dataloader):
            total_steps += opt.input_nc # using input_nc as batch_size proxy if needed, or just 1
            
            model.set_input(data)
            model.optimize_parameters()
            
            if i % 10 == 0:
                print(f"Epoch {epoch}, Iter {i}, Loss: {model.loss.item()}")
        
        model.update_learning_rate()

Found classes: ['human', 'non-human']
Total samples: 90
Total number of parameters: 11742
Model created.


IndexError: index out of range in self

In [ ]:
# 1. Point to the test folder
opt.dataroot = '/Users/ahashganeshamoorthy/Documents/Projects/Radiel/neuro/prototypes/V0/dataset/test'
opt.phase = 'test' 
opt.is_train = False # Important: Tells the dataset loader not to augment/flip data

# 2. Create the Test Loader
test_dataset = MeshDataset(opt)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

# 3. Switch Model to Evaluation Mode
# This turns off "Dropout" and "Batch Norm" updates so results are stable
model.net.eval() 

# 4. Run Inference
print(f"\n--- Running Inference on {len(test_dataset)} files ---\n")
correct = 0

for i, data in enumerate(test_dataloader):
    model.set_input(data) # Load the test shape
    out = model.forward() # Run prediction
    
    # Get the predicted class ID (0 or 1)
    pred_class = out.data.max(1)[1].item()
    true_label = data['label'].item()
    
    # Map ID back to name (assuming alphabetical sort)
    class_names = ['Human', 'Non-Human'] 
    prediction_name = class_names[pred_class]
    true_name = class_names[true_label]
    
    if pred_class == true_label:
        correct += 1
        result = "✅ CORRECT"
    else:
        result = "❌ WRONG"
        
    print(f"File: {i} | Predicted: {prediction_name} | True: {true_name} | {result}")

print(f"\nTotal Accuracy: {correct / len(test_dataset) * 100:.2f}%")